In [1]:
import sys
from pathlib import Path

scripts_path = Path("../scripts").resolve()
sys.path.append(str(scripts_path))

import rq5_function_lib as fl
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import yfinance as yf
import seaborn as sns
import numpy as np
import polars as pl
import plotly.express as px
import plotly.graph_objects as go

## RQ5: E10 - Diesel Price comparison and frquency of anomalies (Diesel price = E10 price//Diesel price > E10 price)

In [3]:
# Paths
# anomalies_path_pc = Path(r"C:\Users\Marvin\Dokumente\Bildung\Uni\dsprojekt\tankerkoenig-data\anomalies")
# parquet_base_pc = Path(r"C:\Users\Marvin\Dokumente\Bildung\Uni\dsprojekt\tankerkoenig-data\prices_parquet")
anomalies_path = Path(r"D:/Tankdaten/anomalies")
parquet_base = Path(r"D:/Tankdaten/prices_parquet")
durations_path = Path("D:/Tankdaten/anomalies/anomaly_durations")
stations_csv = Path(r"C:/Users/Marvin/Dokumente/Bildung/Uni/dsprojekt/tankerkoenig-data/stations/stations.csv")
stations_parquet = Path("stations.parquet")
stats_cache = Path("station_stats_2020_2027.parquet")

# Load files
anomaly_files = sorted(anomalies_path.glob("*_all_anomalies.parquet"))

print(f"Gefundene Anomalie-Dateien: {len(anomaly_files)}")
for f in anomaly_files:
    print(f"  - {f.name}")


Gefundene Anomalie-Dateien: 13
  - 2014_all_anomalies.parquet
  - 2015_all_anomalies.parquet
  - 2016_all_anomalies.parquet
  - 2017_all_anomalies.parquet
  - 2018_all_anomalies.parquet
  - 2019_all_anomalies.parquet
  - 2020_all_anomalies.parquet
  - 2021_all_anomalies.parquet
  - 2022_all_anomalies.parquet
  - 2023_all_anomalies.parquet
  - 2024_all_anomalies.parquet
  - 2025_all_anomalies.parquet
  - 2026_all_anomalies.parquet


In [4]:
# Define the range of years you want to analyze
start_year = 2021
end_year = 2027

# Combine anomalies from how many years you want to see
lfs = []
for year in range(start_year, end_year): 
    year_file = anomalies_path / f"{year}_all_anomalies.parquet"
    if year_file.exists():
        try:
            lf = pl.scan_parquet(year_file)
            # Convert date to datetime (only date part, ignore time and timezone)
            lf = lf.with_columns(
                pl.col("date")
                .str.slice(0, 10)  # Only take YYYY-MM-DD
                .str.to_date()
                .alias("date_parsed")
            )
            lfs.append(lf)
            print(f"{year} queued")
        except Exception as e:
            print(f"X {year} error: {e}")

# 
combined = pl.concat(lfs).collect()
combined = combined.with_columns(
    year=pl.col("date_parsed").dt.year(),
    month=pl.col("date_parsed").dt.month(),
    week=pl.col("date_parsed").dt.week()
)

df_durations = (
    pl.scan_parquet(str(durations_path / "real_anomaly_durations_*.parquet"))
    .with_columns(pl.col("start").dt.year().alias("year"))
    .collect()
)
print(f"\nTotal Anomalies: {combined.height:,} from {start_year}-{end_year-1}")

2021 queued
2022 queued
2023 queued
2024 queued
2025 queued
2026 queued

Total Anomalies: 177,970,378 from 2021-2026


## Anomaly Rate per Month (anomalies / total updates)

In [5]:
fig = fl.plot_anomaly_rate(combined, parquet_base, start_year, end_year)
fig.show()

The plot shows the monthly anomaly rate (share of price updates where Diesel ≥ E10) from 2021 to 2026.

- Before 2022 the rate is low and stable, typically below 10% — Diesel being cheaper than E10 was the norm.
- **March 2022** shows a sharp spike to nearly 100%, coinciding with the outbreak of the Russia-Ukraine war, which caused a sudden surge in crude oil and diesel prices.
- The rate stays elevated throughout 2022 and gradually decreases over 2023–2024 as fuel markets stabilize.
- **Early 2026** shows a second spike, driven by renewed geopolitical tensions in the Middle East (USA/Israel–Iran conflict), pushing diesel prices above E10 again.

## Total Anomalies per year

In [5]:
fig = fl.plot_anomalies_per_year(combined, start_year, end_year)
fig.show()

The bar chart shows the total number of anomaly records per year.

- **2022** dominates by a wide margin, reflecting the near-constant diesel ≥ E10 situation throughout that year.
- 2021 and 2023–2025 show a much lower anomaly count, confirming that the 2022 spike was an exceptional event rather than the new baseline.
- The visible increase in 2026 aligns with the second geopolitical spike already seen in the monthly rate chart.

## Distribution of Price anomalies by hour of day for any year

In [6]:
year = 2022
fig = fl.plot_anomaly_rate_by_hour(parquet_base, year, anomalies_path=anomalies_path)
fig.show()

The plot shows the anomaly rate broken down by hour of the day for the selected year (here: 2022).

- The anomaly rate is consistently near **~80% across all 24 hours**, meaning diesel was equal to or more expensive than E10 at virtually every hour of every day in 2022.
- There is no significant intraday pattern — the anomaly was not a time-of-day effect but a persistent market condition.
- Comparing to a normal year (e.g. 2021) would show a much flatter, lower rate with more hourly variation.

## Top 100 Gas Stations with highest anomalie rate (= anomalies/price updates) between 2020 and 2026

In [7]:
fig, map_df = fl.plot_top_stations_map(stations_csv, stations_parquet, stats_cache, start_year, end_year, anomalies_path=anomalies_path)
fig.show()

The map plots the top 100 gas stations with the highest individual anomaly rates (anomalies / total price updates) between 2021 and 2026.

- The stations are **clustered heavily in eastern Germany**, particularly in Brandenburg, Saxony, and Mecklenburg-Vorpommern.
- This geographic concentration suggests that regional factors — such as proximity to refinery supply routes, regional pricing competition, or lower baseline E10 demand — contribute to a structurally higher diesel ≥ E10 frequency at these locations.
- Western and southern Germany are largely absent from the top 100, indicating more balanced diesel/E10 pricing in those regions.

In [14]:
import importlib; importlib.reload(fl)

<module 'rq5_function_lib' from 'C:\\Users\\marvh\\Documents\\Uni\\dsprojekt\\Data-Science-Projekt\\scripts\\rq5_function_lib.py'>

## Median anomaly duration by year

In [ ]:
fig = fl.plot_median_anomaly_duration(df_durations)
fig.show()

The line chart shows the median duration of a single anomaly episode (in minutes) per year.

- In 2021 the median duration was around **118 minutes (~2 hours)** — anomalies were short-lived, likely caused by brief pricing errors or local fluctuations.
- In **2022** the median jumped to **~864 minutes (~14.4 hours)**, meaning a typical anomaly lasted more than half a day. This reflects the sustained market shift caused by the Russia-Ukraine war: diesel simply stayed more expensive than E10 for long stretches.
- From 2023 onward the median gradually returns toward pre-war levels, indicating that anomaly episodes became shorter and more transient again.

# Anomaly Distribution by year

In [4]:
fig = fl.plot_anomaly_duration_distribution(df_durations)
fig.show()

The box plot shows the full distribution of anomaly durations per year, revealing spread and outliers beyond just the median.

- The **2022 box is dramatically wider and higher** than other years: the interquartile range (IQR) stretches from ~1.9 h (Q1) to ~136 h (Q3), meaning 50% of all anomalies lasted between roughly 2 and 136 hours.
- The median (Q2) sits at ~14.4 h for 2022, confirming the sustained nature of the price inversion.
- Other years show compact boxes close to zero, indicating that outside of 2022 most anomalies were resolved within a few hours.
- Long upper whiskers and outliers in 2022 show that some stations had diesel ≥ E10 for several days or even weeks without interruption.

---

## Results — RQ5: E10 vs. Diesel Price Anomalies (2021–2026)

**Definition:** An anomaly occurs when the diesel price ≥ E10 price at a given station and timestamp. Under normal market conditions diesel is cheaper than E10 due to lower taxation.

---

### Key Findings

**1. Geopolitical shocks dominate anomaly frequency**
The anomaly rate was low and stable before 2022 (<10% monthly). The Russia-Ukraine war (March 2022) caused a near-instant jump to ~100%, persisting for most of the year. A second, smaller spike occurred in early 2026 due to Middle-East tensions. Outside these events, anomaly rates returned to baseline.

**2. Diesel reacts more strongly to crude oil prices than E10**
The anomalies are not a direct consequence of the war itself, but of the **crude oil price spikes** triggered by it. Diesel is produced more directly from crude oil fractions and has a higher sensitivity to oil price changes, while E10 (a petrol blend with 10% ethanol) is partially decoupled through its ethanol component. This means that when oil prices surge sharply, diesel prices rise faster and further than E10 — inverting the usual price relationship.

**3. 2022 was a structural market inversion, not noise**
The median anomaly duration in 2022 (~14.4 h) was ~7× higher than in 2021 (~2 h). Box plot distributions confirm that in 2022 the diesel ≥ E10 condition was sustained for hours to days at a time — not a measurement artifact or brief pricing glitch.

**4. No intraday pattern**
The ~80% hourly anomaly rate in 2022 was uniform across all 24 hours, confirming this was a persistent price level effect rather than a time-of-day pricing strategy.

**5. Regional concentration in eastern Germany**
The top 100 stations by anomaly rate are clustered in eastern Germany (Brandenburg, Saxony, Mecklenburg-Vorpommern). This points to structural regional differences in supply chains, competition, or demand for E10.

---

### Summary

Diesel prices exceeding E10 prices are rare under normal conditions but become near-universal during major crude oil price surges. Geopolitical events (Russia-Ukraine war 2022, Middle-East conflict 2026) acted as triggers by driving up oil prices, to which diesel responds more sensitively than E10. The 2022 oil price shock is the single largest driver of anomalies in the observed period, both in frequency and duration. Regional factors further modulate which stations are most affected.